In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

# --- LOAD DATA ---
df = pd.read_csv("cleaned_patient_with_hospital.csv")

# --- FILTER DISEASE ---
df = df[df["disease_type"].str.lower().str.contains("diarrhea")].copy()

# --- PARSE DATE ---
df["date_disease_start"] = pd.to_datetime(df["date_disease_start"], errors="coerce")
df = df.dropna(subset=["date_disease_start"])

df["year"] = df["date_disease_start"].dt.year
df["month"] = df["date_disease_start"].dt.month

# --- AGGREGATE ---
monthly_cases = (
    df.groupby(["district", "year", "month"])
    .size()
    .reset_index(name="case_count")
)

# Sort properly for lag features
monthly_cases = monthly_cases.sort_values(by=["district", "year", "month"])

# --- FEATURE ENGINEERING ---
monthly_cases["lag_1"] = monthly_cases.groupby("district")["case_count"].shift(1)
monthly_cases["lag_2"] = monthly_cases.groupby("district")["case_count"].shift(2)

monthly_cases["rolling_avg_2"] = monthly_cases[["lag_1", "lag_2"]].mean(axis=1)
monthly_cases["rolling_max_2"] = monthly_cases[["lag_1", "lag_2"]].max(axis=1)

monthly_cases["month_of_year"] = monthly_cases["month"]

# Drop rows with NaN (from lag)
monthly_cases = monthly_cases.dropna()

# --- LABEL CREATION ---
threshold = np.percentile(monthly_cases["case_count"], 75)

monthly_cases["label"] = (monthly_cases["case_count"] > threshold).astype(int)

# --- CREATE DATE FOR SPLIT ---
monthly_cases["date"] = pd.to_datetime(
    monthly_cases["year"].astype(str) + "-" + monthly_cases["month"].astype(str) + "-01"
)

# --- TIME-BASED SPLIT ---
cutoff_index = int(len(monthly_cases) * 0.8)
cutoff_date = monthly_cases["date"].sort_values().iloc[cutoff_index]

train_set = monthly_cases[monthly_cases["date"] < cutoff_date]
test_set  = monthly_cases[monthly_cases["date"] >= cutoff_date]

# --- FEATURES / TARGET ---
features = ["lag_1", "lag_2", "rolling_avg_2", "rolling_max_2", "month_of_year"]
target = "label"

X_train = train_set[features]
y_train = train_set[target]

X_test = test_set[features]
y_test = test_set[target]

# --- MODEL ---
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# --- EVALUATION ---
y_pred = model.predict(X_test)

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# --- PREDICTION (LATEST PER DISTRICT) ---
latest_rows = monthly_cases.sort_values("date").groupby("district").tail(1)

latest_rows["prediction"] = model.predict(latest_rows[features])

# Flag districts
flagged = latest_rows[latest_rows["prediction"] == 1]

print("\nDistricts needing diarrhea medicine:")
print(flagged[["district", "prediction"]])

Precision: 0.8
Recall: 0.8
F1 Score: 0.8
Confusion Matrix:
 [[107   7]
 [  7  28]]
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       114
           1       0.80      0.80      0.80        35

    accuracy                           0.91       149
   macro avg       0.87      0.87      0.87       149
weighted avg       0.91      0.91      0.91       149


Districts needing diarrhea medicine:
            district  prediction
71   bang_khun_thian           1
23         bang_kapi           1
35         bang_khae           1
585         watthana           1
59     bang_kho_laem           1
431           prawet           1
513       suan_luang           1
347       nong_khaem           1
335        nong_chok           1
299      lat_krabang           1
167        chatuchak           1
